# In this tutorial, we present some of the code used in the article entitled 'Entropy-based regularized regression for advanced distribution of relaxation times deconvolution' (DOI: j.jpowsour.2025.236910).

In [ ]:
# import necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import log10, pi
from scipy.optimize import minimize
from bayes_opt import BayesianOptimization
import cvxpy as cp
import os
import pymc as pm
import arviz as az

# custom module import
import compute_DRT

In [2]:
# plot settings
plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=15)
plt.rc('xtick', labelsize=15)
plt.rc('ytick', labelsize=15)

# colors for the figures
CB_color_cycle = ['#377eb8', '#ff7f00', '#4daf4a',
                  '#f781bf', '#a65628', '#984ea3',
                  '#999999', '#e41a1c', '#dede00']

# 1. Generate artificial impedance data based on the Gerischer model

## 1.1 Define the functions

In [3]:
def fct_Z_ZARC_exact(freq_vec, R_inf, R_ct, tau_0):
    '''computes exact impedance for ZARC model.'''
    omega = 2 * pi * freq_vec
    Z_exact = R_inf + R_ct / np.sqrt(1.0 + 1j * omega * tau_0)
    return Z_exact, np.concatenate([Z_exact.real, Z_exact.imag])

def fct_gamma_ZARC_exact(tau_vec, R_ct, tau_0):
    '''computes gamma_exact for the ZARC model.'''
    N_tau = len(tau_vec)
    gamma_exact = np.zeros((N_tau))
    for n in range(N_tau):
        if tau_vec[n]<tau_0:
            gamma_exact[n] = R_ct/np.pi*np.sqrt(tau_vec[n]/(tau_0-tau_vec[n]))
    return gamma_exact

def fct_Z_synth(Z_exact, sigma_n_exp=0.5, random_seed=1225):
    '''adds synthetic noise to the impedance.'''
    np.random.seed(random_seed)
    N_freqs = len(Z_exact)
    noise = np.random.normal(0, sigma_n_exp, N_freqs) + 1j * np.random.normal(0, sigma_n_exp, N_freqs)
    Z_exp = Z_exact + noise
    return Z_exp, np.concatenate([Z_exp.real, Z_exp.imag])

def prob_RR(lambda_0, A, L, Z):
    '''performs ridge regression with L2 regularization.'''
    x = cp.Variable(A.shape[1], nonneg=True)
    objective = cp.Minimize(cp.norm(A @ x - Z, p=2)**2 + lambda_0 * cp.norm(L @ x, p=2)**2)
    cp.Problem(objective).solve(solver=cp.MOSEK)
    return A @ x.value, x.value

def prob_entropy(lambda_0, A, Z):
    '''solves optimization problem with entropy regularization.'''
    x = cp.Variable(A.shape[1], nonneg=True)
    objective = cp.Minimize(cp.norm(A @ x - Z, p=2)**2 - lambda_0 * cp.sum(cp.entr(x)))
    cp.Problem(objective).solve(solver=cp.MOSEK)
    return A @ x.value, x.value

def objective(log10_lambda_0, problem_func, A, Z_exp_re_im, gamma_exact):
    '''objective function for Bayesian optimization.'''
    lambda_reg = 10**log10_lambda_0
    _, x = problem_func(lambda_reg, A, Z_exp_re_im)
    return -log10(np.linalg.norm(x[1:] - gamma_exact, 2))

def perform_bo(problem_name, problem_func, A, Z_exp_re_im, gamma_exact, pbounds, random_state):
    '''performs bayesian optimization to find the optimal lambda.'''
    optimizer = BayesianOptimization(
        f=lambda log10_lambda_0: objective(log10_lambda_0, problem_func, A, Z_exp_re_im, gamma_exact),
        pbounds=pbounds,
        random_state=random_state
    )
    optimizer.maximize(init_points=5, n_iter=40)

    optimal_lambda = 10**optimizer.max['params']['log10_lambda_0']
    Z_hat_BO_opt, x_BO_opt = problem_func(optimal_lambda, A, Z_exp_re_im)

    error_gamma = np.linalg.norm(x_BO_opt[1:] - gamma_exact)
    error_Z = np.linalg.norm(A @ x_BO_opt - Z_exp_re_im)

    print(f'Optimal lambda ({problem_name}): {optimal_lambda}')
    print(f'Gamma error ({problem_name}): {error_gamma}')
    print(f'Z error ({problem_name}): {error_Z}\n')

    return optimal_lambda, Z_hat_BO_opt, x_BO_opt

def sample_posterior(A, Z, lambda_0, L=None, sigma_n_exp=0.5, draws=4000, method='entropy', upper_bound=10, x_init=None, sampler='NUTS'):
    '''samples posterior using pymc.'''
    x_shape = A.shape[1]
    eps = 1e-8  # small constant for regularization term

    with pm.Model() as model:

        # define the prior for x using Uniform distributions with a small positive lower bound
        x = pm.Uniform('x', lower=eps, upper=upper_bound, shape=x_shape)
        
        # likelihood: observed data modeled as Normal with mean A @ x
        mu = pm.math.dot(A, x)
        Z_obs = pm.Normal('Z_obs', mu=mu, sigma=sigma_n_exp, observed=Z)
        
        if method == 'entropy':
            # regularization term: entropy
            entropy_term = 0.5*(lambda_0 / sigma_n_exp**2) * pm.math.sum(x * pm.math.log(x))
            pm.Potential('entropy', -entropy_term)
            
            # # add a tiny L2 regularization term
            # l2_reg = -0.5 * eps * pm.math.sum(pm.math.sqr(x))
            # pm.Potential('tiny_l2_regularization', l2_reg)

        elif method == 'RR':
            if L is None:
                raise ValueError('L matrix is required for RR method.')
            # regularization term: L2 norm (ridge regression)
            l2_reg = 0.5 * (lambda_0 / sigma_n_exp**2) * pm.math.sum(pm.math.sqr(pm.math.dot(L, x)))
            pm.Potential('l2_regularization', -l2_reg)

        else:
            raise ValueError("invalid method specified. choose 'entropy' or 'RR'.")   

        # use MAP estimates as starting point if provided
        if x_init is not None:
            # ensure x_init is within bounds
            x_init = np.clip(x_init, eps, upper_bound)
            initvals = {'x': x_init}
        else:
            initvals = None

        # instantiate the step method
        if sampler == 'NUTS':
            step = pm.NUTS(target_accept=0.95)
        elif sampler == 'Metropolis':
            step = pm.Metropolis()
        elif sampler == 'Slice':
            step = pm.Slice()
        else:
            raise ValueError("invalid sampler specified. choose 'NUTS', 'Metropolis', or 'Slice'.")

        # sampling configuration
        trace = pm.sample(
            draws=draws,
            tune=int(draws * 0.5),
            step=step,
            initvals=initvals,
            return_inferencedata=True,
            chains=4,
            cores=min(4, os.cpu_count()),
            progressbar=True
        )

    # generate sampling diagnostics
    print(f'Sampling diagnostics for method \'{method}\', sampler \'{sampler}\':')
    summary_df = az.summary(trace, var_names=['x'], round_to=2)
    print(summary_df)
    summary_df.to_csv('./bo_sampling_summary.csv')
    az.plot_trace(trace, var_names=['x'])
    plt.close()

    return trace

def extract_samples(trace, varname):
    '''extracts and reshapes samples from the pymc trace.'''
    var_samples = trace.posterior[varname].stack(sample=('chain', 'draw')).values
    var_samples = var_samples.transpose()  # shape: (num_samples, variable_dimension)
    return var_samples

## 1.2 Define the exact and experimental impedances

In [4]:
# set up parameters
N_freqs = 81
freq_vec = np.logspace(-2, 6, num=N_freqs)
R_inf = 10
R_ct = 50
tau_0 = 0.01

# calculate exact impedance and add synthetic noise
Z_exact, _ = fct_Z_ZARC_exact(freq_vec, R_inf, R_ct, tau_0)
Z_exp, Z_exp_re_im = fct_Z_synth(Z_exact)

## 1.3 Define the exact DRT

In [16]:
# fine grid
N_tau_fine = 801
tau_vec_fine = np.logspace(-6, 2, num=N_tau_fine)
log_tau_vec_fine = np.log(tau_vec_fine)
gamma_exact_fine = fct_gamma_ZARC_exact(tau_vec_fine, R_ct, tau_0)

N_tau = 81
tau_vec = np.logspace(-6, 2, num=N_tau)
log_tau_vec = np.log(tau_vec)
gamma_exact = fct_gamma_ZARC_exact(tau_vec, R_ct, tau_0)

## 1.4 Plot the exact and experimental impedances

In [17]:
# plot Nyquist diagram
plt.plot(Z_exact.real, -Z_exact.imag, 'b--', linewidth=2, label='exact')
plt.plot(Z_exp.real, -Z_exp.imag, 'ro', markersize=7, label='exp')
plt.axis('scaled')
plt.gca().set_aspect('equal', adjustable='box')
plt.legend(frameon=False, fontsize=15)
plt.xlabel(r'$Z_{\rm re}/\Omega$', fontsize=20)
plt.ylabel(r'$-Z_{\rm im}/\Omega$', fontsize=20)
fig = plt.gcf()
fig.set_size_inches(6.472, 4)
plt.show()

# 2. Deconvolve the DRTs with ridge regression (RR) and with the entropy

## 2.1 Compute the discretization and differentiation matrices

In [18]:
# compute matrices A and L
A_complex = compute_DRT.compute_A_complex(freq_vec, log_tau_vec)
A = np.vstack((np.hstack((np.ones((N_freqs, 1)), A_complex.real)),
               np.hstack((np.zeros((N_freqs, 1)), A_complex.imag))))
L2 = np.hstack((np.zeros((N_tau-2, 1)), compute_DRT.compute_L2(log_tau_vec)))

## 2.2 Recover the DRTs using Bayesian optimization

In [20]:
# perform bayesian optimization
pbounds = {'log10_lambda_0': (-6, 2)}
random_state = 42
optimal_lambda_RR, Z_hat_RR, x_RR_opt = perform_bo(
    'RR', lambda l, A, Z: prob_RR(l, A, L2, Z), A, Z_exp_re_im, gamma_exact, pbounds, random_state)
optimal_lambda_entropy, Z_hat_entropy, x_entropy_opt = perform_bo(
    'entropy', prob_entropy, A, Z_exp_re_im, gamma_exact, pbounds, random_state)

# compute the errors
print('DRT error entropy', np.linalg.norm(x_entropy_opt[1:] - gamma_exact))
print('Impedance error entropy', np.linalg.norm(A @ x_entropy_opt - Z_exp_re_im))
print('DRT error RR', np.linalg.norm(x_RR_opt[1:] - gamma_exact)])
print('Impedance error RR', np.linalg.norm(A @ x_RR_opt - Z_exp_re_im))

|   iter    |  target   | log10_... |
-------------------------------------
| 1         | -1.64     | -3.004    |
| 2         | -1.519    | 1.606     |
| 3         | -1.404    | -0.144    |
| 4         | -1.437    | -1.211    |
| 5         | -1.867    | -4.752    |
| 6         | -1.406    | -0.5362   |
| 7         | -1.403    | -0.2962   |
| 8         | -1.403    | -0.2435   |
| 9         | -1.403    | -0.2379   |
| 10        | -1.403    | -0.2337   |
| 11        | -1.403    | -0.2302   |
| 12        | -1.403    | -0.2273   |
| 13        | -1.403    | -0.2253   |
| 14        | -1.403    | -0.263    |
| 15        | -1.403    | -0.21     |
| 16        | -1.403    | -0.216    |
| 17        | -1.403    | -0.2646   |
| 18        | -1.403    | -0.2141   |
| 19        | -1.403    | -0.2179   |
| 20        | -1.403    | -0.2607   |
| 21        | -1.403    | -0.2163   |
| 22        | -1.403    | -0.219    |
| 23        | -1.403    | -0.2571   |
| 24        | -1.403    | -0.2176   |
| 25        

# 3. Sampling step

## 3.1 Sample the DRT from RR

In [ ]:
# perform pymc sampling
trace_RR = sample_posterior(
    A, Z_exp_re_im, optimal_lambda_RR,
    L=L2, sigma_n_exp=0.5, draws=10000, method='RR',
    upper_bound=1000, x_init=x_RR_opt, sampler='NUTS')

## 3.2 Sample the DRT from the entropy

In [25]:
# for the entropy method with tiny L2 regularization
trace_entropy = sample_posterior(
    A, Z_exp_re_im, optimal_lambda_entropy,
    L=None, sigma_n_exp=0.5, draws=4000, method='entropy',
    upper_bound=1000, x_init=x_entropy_opt, sampler='NUTS'
)

Multiprocess sampling (4 chains in 4 jobs)
NUTS: [x]


Output()

Sampling 4 chains for 2_000 tune and 4_000 draw iterations (8_000 + 16_000 draws total) took 10076 seconds.


Sampling diagnostics for method 'entropy', sampler 'NUTS':
       mean    sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  \
x[0]   9.91  0.13    9.67    10.17        0.0      0.0  14483.38  12528.18   
x[1]   0.66  0.50    0.00     1.57        0.0      0.0  12845.22   6902.62   
x[2]   0.53  0.40    0.00     1.25        0.0      0.0   7092.03   4414.12   
x[3]   0.53  0.40    0.00     1.25        0.0      0.0  10048.32   5626.43   
x[4]   0.53  0.40    0.00     1.24        0.0      0.0  10306.79   5761.46   
...     ...   ...     ...      ...        ...      ...       ...       ...   
x[77]  0.56  0.45    0.00     1.37        0.0      0.0   9254.54   4916.72   
x[78]  0.60  0.47    0.00     1.44        0.0      0.0   7976.97   4471.17   
x[79]  0.64  0.49    0.00     1.53        0.0      0.0  12334.10   6275.48   
x[80]  0.67  0.51    0.00     1.58        0.0      0.0  10722.09   5101.59   
x[81]  0.75  0.55    0.00     1.74        0.0      0.0  13629.81   6807.26   

    

## 3.3 Compute the samples statistics

In [26]:
# RR
x_samples_RR = extract_samples(trace_RR, 'x')
x_mean_RR = np.mean(x_samples_RR, axis=0)
x_lower_RR = np.percentile(x_samples_RR, 2.5, axis=0)
x_upper_RR = np.percentile(x_samples_RR, 97.5, axis=0)
np.save('./x_samples_RR.npy', x_samples_RR)

# entropy
x_samples_entropy = extract_samples(trace_entropy, 'x')
x_mean_entropy = np.mean(x_samples_entropy, axis=0)
x_lower_entropy = np.percentile(x_samples_entropy, 2.5, axis=0)
x_upper_entropy = np.percentile(x_samples_entropy, 97.5, axis=0)
np.save('./x_samples_entropy.npy', x_samples_entropy)

# 4. Plots the sampled DRTs

## 4.1 Plot the maximum-a-posteriori (MAP) and 95% credible intervals for RR (Figure 4 (a) in the manuscript)

In [ ]:
percentiles_RR = np.percentile(x_samples_RR[:,1:], [5, 25, 50, 75, 95], axis=0)
plt.fill_between(tau_vec, percentiles_RR[0], percentiles_RR[4], alpha=0.2, color=CB_color_cycle[2], label='5th-95th')
plt.fill_between(tau_vec, percentiles_RR[1], percentiles_RR[3], alpha=0.4, color=CB_color_cycle[2], label='25th-75th')
plt.semilogx(tau_vec, percentiles_RR[2], '-', color=CB_color_cycle[2], label='median', linewidth=2) # change MAP to dots
plt.semilogx(tau_vec, x_RR_opt[1:], '.', color=CB_color_cycle[2], label='MAP', markersize=8)
plt.semilogx(tau_vec_fine, gamma_exact_fine, 'k--', linewidth=2, label='exact')
plt.xlabel(r'$\tau/\rm s$', fontsize = 20)
plt.ylabel(r'$\gamma/\Omega$', fontsize = 20)
plt.xlim(1e-6, 1e2)
plt.ylim(0, 30)
plt.yticks(np.arange(0, 30.1, 10))
plt.xticks([1e-6, 1e-4, 1e-2, 1e0, 1e2])
fig = plt.gcf()
fig.set_size_inches(6.472, 4)
plt.show()

## 4.2 Plot the MAP and 95% credible intervals for the entropy (Figure 4 (b))

In [ ]:
percentiles_entropy = np.percentile(x_samples_entropy[:,1:], [5, 25, 50, 75, 95], axis=0)
plt.fill_between(tau_vec, percentiles_entropy[0], percentiles_entropy[4], alpha=0.2, color=CB_color_cycle[3], label='5th-95th')
plt.fill_between(tau_vec, percentiles_entropy[1], percentiles_entropy[3], alpha=0.4, color=CB_color_cycle[3], label='25th-75th')
plt.semilogx(tau_vec, percentiles_entropy[2], '-', color=CB_color_cycle[3], label='median', linewidth=2) # change MAP to dots
plt.semilogx(tau_vec, x_entropy_opt[1:], '.', color=CB_color_cycle[3], label='MAP', markersize=8)
plt.semilogx(tau_vec_fine, gamma_exact_fine, 'k--', linewidth=2, label='exact')
plt.xlabel(r'$\tau/\rm s$', fontsize = 20)
plt.ylabel(r'$\gamma/\Omega$', fontsize = 20)
plt.xlim(1e-6, 1e2)
plt.ylim(0, 30)
plt.yticks(np.arange(0, 30.1, 10))
plt.xticks([1e-6, 1e-4, 1e-2, 1e0, 1e2])
fig = plt.gcf()
fig.set_size_inches(6.472, 4)
plt.show()